In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Subset, DataLoader, Dataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Configurações Básicas
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Treinando na: {device}")
BATCH_SIZE = 32
EPOCHS = 50

# Dataset
caminho_arquivo = './malimg.npz'
arquivo_npz = np.load(caminho_arquivo, allow_pickle=True)
dados_brutos = arquivo_npz['arr']

class MalimgNumpyDataset(Dataset):
    def __init__(self, data_array, transform=None):
        self.images = data_array[:, 0]
        mapa_familias = {
            7: 6,
            21: 20,
        }
        self.labels = np.array([mapa_familias.get(lbl, lbl) for lbl in data_array[:, 1]])
        self.transform = transform

        self.classes = np.unique(self.labels)
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        label_nome = self.labels[idx]
        label_idx = self.class_to_idx[label_nome]

        if hasattr(img_array, 'dtype') and img_array.dtype != np.uint8:
            img_array = img_array.astype(np.uint8)

        img_pil = Image.fromarray(img_array)

        if self.transform:
            img_pil = self.transform(img_pil)

        return img_pil, torch.tensor(label_idx, dtype=torch.long)

transform_stats = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor()
])

dataset_stats = MalimgNumpyDataset(data_array=dados_brutos, transform=transform_stats)

print(f"Total de imagens processadas: {len(dataset_stats)}")
numeroClasses = len(dataset_stats.classes)
print(f"Famílias de malware encontradas: {numeroClasses}")

labels_totais = [dataset_stats.class_to_idx[label] for label in dataset_stats.labels]
indices_totais = list(range(len(dataset_stats)))

idx_train, idx_test, y_train, y_test = train_test_split(
    indices_totais, 
    labels_totais, 
    test_size=0.20, 
    random_state=42
)

subset_treino_stats = Subset(dataset_stats, idx_train)
loader_stats = DataLoader(subset_treino_stats, batch_size=64, shuffle=False, num_workers=2)

soma = 0.0
soma_quadrados = 0.0
total_pixels = 0

for imgs, _ in loader_stats:
    imgs_d = imgs.double()
    soma += imgs_d.sum().item()
    soma_quadrados += (imgs_d ** 2).sum().item()
    total_pixels += imgs_d.numel()

media_malimg = soma / total_pixels
std_malimg = np.sqrt((soma_quadrados / total_pixels) - (media_malimg ** 2))
print(f"Malimg (Treino) -> Média: {media_malimg:.4f} | Desvio Padrão: {std_malimg:.4f}")

transformacao_treino = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomRotation(degrees=10, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomApply([
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))], p=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.10), fill=0),
    transforms.RandomCrop(224, padding=4, padding_mode='edge'),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[media_malimg, media_malimg, media_malimg],
        std=[std_malimg, std_malimg, std_malimg]
    )
])

transformacao_teste = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[media_malimg, media_malimg, media_malimg],
        std=[std_malimg, std_malimg, std_malimg]
    )
])

dataset_para_treino = MalimgNumpyDataset(data_array=dados_brutos, transform=transformacao_treino)
dataset_para_teste = MalimgNumpyDataset(data_array=dados_brutos, transform=transformacao_teste)

train_data = Subset(dataset_para_treino, idx_train)
test_data = Subset(dataset_para_teste, idx_test)

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

print(f"Divisão -> Treino: {len(train_data)} | Teste: {len(test_data)}")

class NormalizacaoL2(nn.Module):
    def __init__(self, scale=20.0):
        super().__init__()
        self.scale = scale

    def forward(self, x):
        x_norm = F.normalize(x, p=2.0, dim=1, eps=1e-12)
        
        return x_norm * self.scale
    
# ResNet34
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

num_ftrs = model.fc.in_features

model.fc = nn.Sequential(
    nn.BatchNorm1d(num_ftrs),
    NormalizacaoL2(scale=20.0),
    nn.Dropout(p=0.6),
    nn.Linear(num_ftrs, numeroClasses)
)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=3,
    min_lr=1e-6
)

# Funções de Treino e Teste
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    loss_treino = 0
    corretos_treino = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        loss_treino += loss.item()
        corretos_treino += (pred.argmax(1) == y).type(torch.float).sum().item()

    acc_treino = (corretos_treino / len(dataloader.dataset)) * 100
    loss_medio_treino = loss_treino / len(dataloader)
    print(f"  Acc: {acc_treino:.2f}% | Loss: {loss_medio_treino:.4f}")
    return acc_treino, loss_medio_treino

def test_loop(dataloader, model, loss_fn):
    model.eval()
    loss_test = 0
    corretos_test = 0
    todas_preds = []
    todos_labels = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss_test += loss_fn(pred, y).item()

            preds_batch = pred.argmax(1)
            corretos_test += (preds_batch == y).type(torch.float).sum().item()

            todas_preds.extend(preds_batch.cpu().numpy())
            todos_labels.extend(y.cpu().numpy())

    acc_test = (corretos_test / len(dataloader.dataset)) * 100
    loss_medio_test = loss_test / len(dataloader)

    precision_test = precision_score(todos_labels, todas_preds, average='macro', zero_division=0) * 100
    recall_test = recall_score(todos_labels, todas_preds, average='macro', zero_division=0) * 100
    f1_test = f1_score(todos_labels, todas_preds, average='macro') * 100

    print(f"  Acc: {acc_test:.2f}% | Precision: {precision_test:.2f}% | Recall: {recall_test:.2f}% | F1: {f1_test:.2f}%")
    return acc_test, loss_medio_test, f1_test, precision_test, recall_test

historico_train_acc = []
historico_train_loss = []
historico_test_acc = []
historico_test_loss = []
historico_test_f1 = []

bestPrecision = 0.0
patienceCounter = 0
patience = 30

# Ciclo de Épocas
print("\nIniciando Treino da ResNet34")
for t in range(EPOCHS):
    print(f"\nÉpoca {t+1}\n-------------------------------")

    print("Treino:")
    train_acc, train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)

    print("Teste:")
    test_acc, test_loss, test_f1, test_precision, test_recall = test_loop(test_dataloader, model, loss_fn)

    historico_train_acc.append(train_acc)
    historico_train_loss.append(train_loss)
    historico_test_acc.append(test_acc)
    historico_test_loss.append(test_loss)
    historico_test_f1.append(test_f1)
    scheduler.step(test_f1)

    if test_acc > bestPrecision:
        bestPrecision = test_acc
        patienceCounter = 0
        torch.save(model.state_dict(), "pesoscnn.pth")
    else:
        patienceCounter += 1

    if patienceCounter >= patience:
        print("\nEarly stopping ativado!")
        break

print("\nTreinamento Concluído!")

# Avaliação Final e Gráficos
model.load_state_dict(torch.load("pesoscnn.pth", weights_only=True))

print("\nResultados Finais:")
precisao_teste_final, loss_teste_final, f1_teste_final, precision_final, recall_final = test_loop(test_dataloader, model, loss_fn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
epocas = range(1, len(historico_train_acc) + 1)

melhor_epoca_idx = historico_test_acc.index(bestPrecision)
melhor_epoca_num = melhor_epoca_idx + 1
melhor_loss_test = historico_test_loss[melhor_epoca_idx]
melhor_f1_test = historico_test_f1[melhor_epoca_idx]

texto_anotacao = f"Epoch: {melhor_epoca_num}\nAcc: {bestPrecision:.2f}%\nF1: {melhor_f1_test:.2f}%\nLoss: {melhor_loss_test:.4f}"
estilo_caixa = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='black')

ax1.plot(epocas, historico_train_loss, label='Train Loss', color='blue', linewidth=2)
ax1.plot(epocas, historico_test_loss, label='Test Loss', color='green', linewidth=2)

ax1.annotate(texto_anotacao,
             xy=(melhor_epoca_num, melhor_loss_test),
             xytext=(0, 50), textcoords='offset points',
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
             fontsize=10, color='black', weight='bold', ha='center', bbox=estilo_caixa)

ax1.set_title("Loss x Epochs")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

ax2.plot(epocas, historico_train_acc, label='Train Acc', color='blue', linewidth=2)
ax2.plot(epocas, historico_test_acc, label='Test Acc', color='green', linewidth=2)

ax2.plot(epocas, historico_test_f1, label='Test F1', color='purple', linewidth=2, linestyle='--')

ax2.annotate(texto_anotacao,
             xy=(melhor_epoca_num, bestPrecision),
             xytext=(0, -60), textcoords='offset points',
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
             fontsize=10, color='black', weight='bold', ha='center', bbox=estilo_caixa)

ax2.set_title("Accuracy & F1-Score x Epochs")
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Percentage (%)")
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

texto_titulo = f"Results (Malimg)\nTest Acc: {precisao_teste_final:.2f}% | Test F1: {f1_teste_final:.2f}%"
plt.suptitle(texto_titulo, fontsize=16, fontweight='bold', color='black')

plt.tight_layout()

plt.savefig('BestVersionGroupedFamilies.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
model.eval()
y_true_final = []
y_pred_final = []

with torch.no_grad():
    for X, y in test_dataloader:
        X = X.to(device)
        preds = model(X).argmax(1)
        y_pred_final.extend(preds.cpu().numpy())
        y_true_final.extend(y.numpy())

cm = confusion_matrix(y_true_final, y_pred_final)

fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset_para_teste.classes)
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical', values_format='d', colorbar=False)

plt.title(f"Matriz de Confusão - ResNet34 (Malimg)\nAcc: {precisao_teste_final:.2f}% | F1: {f1_teste_final:.2f}%", fontsize=16, fontweight='bold')
plt.xlabel("Classe Predita", fontsize=12)
plt.ylabel("Classe Real", fontsize=12)
plt.tight_layout()

plt.savefig('matriz_confusao.png', dpi=300, bbox_inches='tight')
plt.show()